In [ ]:
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document

from llm_llamacpp import embedder

/tmp/ipykernel_58707/2214396459.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [2]:
docs = [
    Document(page_content='Civil War is a major possible catastrophic problem in Skyrim'),
    Document(page_content="Dragons are a huge problem and an unprecedented danger that threatens Skyrim's very existence"),
    Document(page_content='A Civil War would greatly affect and harm the common folks of Skyrim, leading to great losses of live and livelihoods This is another problem'),
    Document(page_content='Vampires are a grave problem in Skyrim, and have become a danger to humanity, especiallu through attempts at blocking out the sun itself'),
    Document(page_content='Skyrim is in great risk of a Civil War breaking out, which would be a huge problem')
]

# Purposefully repeated documents about civil war to see MMR's use cases

In [3]:
vectorstore = FAISS.from_documents(
    documents=docs,
    embedding=embedder
)

In [4]:
retriever_mmr = vectorstore.as_retriever(
    search_type='mmr',
    search_kwargs={
        'k':3,
        'lambda_mult':0.5
    }
)

# lambda_mult of 1 makes MMR behave as regular semantic search. So comparing their differences
retriever_usual = vectorstore.as_retriever(
    search_type='similarity',
    search_kwargs={
        'k':3,
        'lambda_mult':1
    }
)

In [5]:
query = 'Instruct: Given a query, retrieve relevant passages that answer the query.\nQuery: "What are the different problems faced by Skyrim"'

results_mmr = retriever_mmr.invoke(query)
results_usual = retriever_usual.invoke(query)

In [6]:
print('According to MMR, the retrieved documents are:')
for i, doc in enumerate(results_mmr):
    print(f'Document no {i+1}\n{doc}', end='\n\n')

According to MMR, the retrieved documents are:
Document no 1
page_content='A Civil War would greatly affect and harm the common folks of Skyrim, leading to great losses of live and livelihoods This is another problem'

Document no 2
page_content='Civil War is a major possible catastrophic problem in Skyrim'

Document no 3
page_content='Dragons are a huge problem and an unprecedented danger that threatens Skyrim's very existence'



In [7]:
print('According to usual semantic search, the retrieved documents are:')
for i, doc in enumerate(results_usual):
    print(f'Document no {i+1}\n{doc}', end='\n\n')

According to usual semantic search, the retrieved documents are:
Document no 1
page_content='A Civil War would greatly affect and harm the common folks of Skyrim, leading to great losses of live and livelihoods This is another problem'

Document no 2
page_content='Civil War is a major possible catastrophic problem in Skyrim'

Document no 3
page_content='Skyrim is in great risk of a Civil War breaking out, which would be a huge problem'



In [8]:
temp_results = vectorstore.similarity_search_with_score(query=query, k=5)

for doc, score in temp_results:
    print(score, doc.page_content)

0.61832523 A Civil War would greatly affect and harm the common folks of Skyrim, leading to great losses of live and livelihoods This is another problem
0.6818479 Civil War is a major possible catastrophic problem in Skyrim
0.84595156 Skyrim is in great risk of a Civil War breaking out, which would be a huge problem
0.9288913 Dragons are a huge problem and an unprecedented danger that threatens Skyrim's very existence
0.96976477 Vampires are a grave problem in Skyrim, and have become a danger to humanity, especiallu through attempts at blocking out the sun itself


#### OK MMR is just not the right tool for this example, and vice versa. There will prolly be more relevant embeddings to use MMR in though, and I will use them then